# Glossary Formatter

This notebook parses `data/glossary.md` into a consolidated glossary table and computes in-notebook summaries/visualizations.

Exports:
- `outputs/glossary/glossary.tsv` (with `source_domain` and `tier` columns)
- `outputs/glossary/glossary_tier_summary.txt`

---

## New Implementation — Parsing `glossary.md`

Parses `data/glossary.md` and writes a single consolidated TSV:

- `outputs/glossary/glossary.tsv`

Summary statistics and the example-share chart are computed below without writing additional files.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from urllib.parse import unquote

import matplotlib.pyplot as plt
import pandas as pd

WORKDIR = Path.cwd()
if not (WORKDIR / 'data').exists():
    WORKDIR = WORKDIR.parent
GLOSSARY_MD_PATH = WORKDIR / 'data' / 'glossary.md'
OUTPUT_DIR = WORKDIR / 'outputs' / 'glossary'
OUTPUT_TSV = OUTPUT_DIR / 'glossary.tsv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GLOSSARY_MD_PATH


In [ ]:
def _extract_link_url(text: str) -> str:
    m = re.search(r'\]\((https?://[^)]+)\)', text)
    return unquote(m.group(1).strip()) if m else ''


def _parse_label(block: str, label: str) -> str:
    m = re.search(rf'_{re.escape(label)}_:\s*(.*?)(?=\s*_[^_]+_:|$)', block, re.S)
    if not m:
        return ''
    return re.sub(r'\s+', ' ', m.group(1)).strip()


def parse_glossary_md(path: Path) -> pd.DataFrame:
    raw = path.read_text(encoding='utf-8', errors='ignore')
    # Normalize backslash line-continuations (soft-wrap artifacts from HTML source)
    raw = re.sub(r'\\\n\s*', '\n', raw)

    # Split into per-term blocks at each **term** heading
    entries = re.split(r'\n(?=\*\*)', raw)

    rows: list[dict] = []
    for entry in entries:
        term_m = re.match(r'\*\*([^*]+)\*\*', entry.strip())
        if not term_m:
            continue
        term = term_m.group(1).strip()

        sf_m = re.search(r'_Surface forms_:\s*([^\n]+)', entry)
        surface_forms = sf_m.group(1).strip() if sf_m else ''

        persona_block_m = re.search(
            r'_Persona/In-Group_:.*?(?=\nDescription|\nExample|\Z)', entry, re.S
        )
        persona_block = persona_block_m.group(0) if persona_block_m else ''
        persona = _parse_label(persona_block, 'Persona/In-Group')
        covert_meaning = _parse_label(persona_block, 'Covert (in-group) meaning')
        dogwhistle_type = _parse_label(persona_block, 'Type')
        register = _parse_label(persona_block, 'Register')

        desc_header_m = re.search(r'Description \(from [^\n]+', entry)
        description_source = _extract_link_url(desc_header_m.group(0)) if desc_header_m else ''
        desc_body_m = re.search(
            r'Description \(from [^\n]+\n(.*?)(?=\nExample context|\Z)', entry, re.S
        )
        description = (
            re.sub(r'\s+', ' ', desc_body_m.group(1)).strip() if desc_body_m else ''
        )

        examples, ex_sources, ex_speakers, ex_dates = [], [], [], []
        for ex_m in re.finditer(
            r'Example context \(in ([^\n]+)\n(.*?)(?=\nExample context|\Z)', entry, re.S
        ):
            source_url = _extract_link_url(ex_m.group(1))
            body = ex_m.group(2)
            speaker = _parse_label(body, 'Speaker')
            date = _parse_label(body, 'Date')
            ex_text = re.sub(r'_Speaker_:.*', '', body, flags=re.S).strip()
            ex_text = re.sub(r'\s+', ' ', ex_text).strip()
            if ex_text:
                examples.append(ex_text)
                ex_sources.append(source_url)
                ex_speakers.append(speaker)
                ex_dates.append(date)

        rows.append(
            {
                'term': term,
                'surface_forms': surface_forms,
                'persona_in_group': persona,
                'covert_meaning': covert_meaning,
                'type': dogwhistle_type,
                'register': register,
                'description': description,
                'description_source': description_source,
                'example_count': len(examples),
                'examples': ' || '.join(examples),
                'example_sources': ' ; '.join(s for s in ex_sources if s),
                'example_speakers': ' ; '.join(s for s in ex_speakers if s),
                'example_dates': ' ; '.join(s for s in ex_dates if s),
            }
        )

    return pd.DataFrame(rows)


glossary_df = parse_glossary_md(GLOSSARY_MD_PATH)
glossary_df.to_csv(OUTPUT_TSV, sep='\t', index=False)

print(f'Wrote {len(glossary_df):,} term rows  →  {OUTPUT_TSV}')
glossary_df.head(5)


---

## Provenance Tier Classification

Classifies each entry by the credibility tier of its description source domain:

| Tier | Domain types |
|------|-------------|
| **1** | Peer-reviewed journals, major news outlets (NYT/WaPo/Guardian/Vox/FiveThirtyEight/Slate), civil-rights watchdogs (ADL/AJC/NCCM), institutional/academic sources |
| **2** | Advocacy orgs, progressive media, subject-matter blogs — credible but below Tier 1 standards |
| **3** | Crowd-edited wikis (RationalWiki, Wikipedia) — high coverage, lower sourcing rigour |
| **unknown** | Domains not in any tier set; printed as warnings for manual review |

Adds `source_domain` and `tier` to the in-memory `glossary_df` and overwrites `glossary.tsv`.

In [ ]:
from urllib.parse import urlparse

# Tier 1: peer-reviewed journals, major mainstream outlets, civil-rights orgs,
# and institutional/academic sources with editorial oversight.
TIER_1_DOMAINS = {
    "link.springer.com", "www.tandfonline.com", "www.taylorfrancis.com", "brill.com",
    "www.nytimes.com", "www.washingtonpost.com", "www.theguardian.com", "www.vox.com",
    "fivethirtyeight.com", "slate.com", "prospect.org", "foreignpolicy.com",
    "www.adl.org", "www.ajc.org", "www.nccm.ca", "antisemitism.org.uk",
    "ianhaneylopez.com", "contemporaryrhetoric.com", "today.tamu.edu",
    "www.law.cuny.edu", "s-usih.org"
}

# Tier 2: advocacy orgs, progressive media, subject-matter blogs — credible
# but without Tier 1 editorial standards or peer review.
TIER_2_DOMAINS = {
    "colorofchange.org", "queervegan.com", "everydayfeminism.com",
    "thedemlabs.org", "blmgrassroots.org", "medium.com", "helenldecruz.medium.com",
    "www.theroot.com", "theconversation.com", "politicalresearch.org",
    "forward.com", "www.dailykos.com", "talkingpointsmemo.com",
    "www.salon.com", "nymag.com", "www.sapiens.org", "www.patheos.com",
    "www.usnews.com", "www.newsweek.com", "www.thedailybeast.com",
    "www.csmonitor.com", "www.latimes.com", "www.haaretz.com",
    "www.jta.org", "storyful.com", "hyperallergic.com", "money.cnn.com",
    "www.ourspectrum.com", "www.cpreview.org", "electionsos.com",
    "jacksonfreepress.com", "papers.ssrn.com"
}

# Tier 3: crowd-edited wikis — broad coverage but lower sourcing rigour.
TIER_3_DOMAINS = {"rationalwiki.org", "en.wikipedia.org"}

# These entries link to google.com/books, verified to resolve to
# López (2014) "Dog Whistle Politics" (OUP) — treated as Tier 1.
TIER_1_OVERRIDES = {
    "affirmative action", "gangbanger", "freedom of association", "food stamp president",
}


def _extract_domain(url: str) -> str:
    if not url:
        return ""
    try:
        return urlparse(url.strip()).netloc.lower()
    except Exception:
        return ""


def _classify_tier(term: str, domain: str) -> str:
    if not domain:
        return "unknown"
    if domain in TIER_1_DOMAINS:
        return "1"
    if domain in TIER_2_DOMAINS:
        return "2"
    if domain in TIER_3_DOMAINS:
        return "3"
    if domain == "www.google.com" and term in TIER_1_OVERRIDES:
        return "1"
    return "unknown"

In [ ]:
OUTPUT_SUMMARY = OUTPUT_DIR / 'glossary_tier_summary.txt'

glossary_df['source_domain'] = glossary_df['description_source'].map(_extract_domain)
glossary_df['tier'] = glossary_df.apply(
    lambda row: _classify_tier(row['term'], row['source_domain']), axis=1
)

# Warn on genuinely unknown domains (non-empty URL that matched no tier)
unknown_mask = (glossary_df['tier'] == 'unknown') & (glossary_df['source_domain'] != '')
for dom in sorted(glossary_df.loc[unknown_mask, 'source_domain'].unique()):
    terms = glossary_df.loc[glossary_df['source_domain'] == dom, 'term'].tolist()
    print(f"WARNING: unknown domain '{dom}' — {len(terms)} term(s): {', '.join(terms)}")

glossary_df.to_csv(OUTPUT_TSV, sep='\t', index=False)
print(f'Updated {len(glossary_df):,} rows → {OUTPUT_TSV}')

# --- Summary report ---
lines = ['=' * 60, 'GLOSSARY PROVENANCE TIER SUMMARY', '=' * 60, '']

tier_counts = glossary_df['tier'].value_counts().reindex(['1', '2', '3', 'unknown'], fill_value=0)
lines += ['Total entries per tier', '-' * 30]
for tier, count in tier_counts.items():
    lines.append(f'  Tier {tier}: {count:>4d} entries')
lines += [f'  TOTAL  : {len(glossary_df):>4d} entries', '']

tier3_df = glossary_df[glossary_df['tier'] == '3']
lines += ['Tier 3 — domain breakdown', '-' * 30]
for dom, cnt in tier3_df['source_domain'].value_counts().items():
    lines.append(f'  {dom}: {cnt}')
lines.append('')

unknown_df = glossary_df[glossary_df['tier'] == 'unknown']
lines += ['Unknown — domain breakdown', '-' * 30]
for dom, cnt in unknown_df['source_domain'].value_counts().items():
    lines.append(f'  {dom if dom else "(no URL)"}: {cnt}')
lines.append('')

lines += ['Per persona/in-group × tier entry counts', '-' * 50]
exploded = glossary_df.copy()
exploded['persona_split'] = exploded['persona_in_group'].str.split(' / ')
exploded = exploded.explode('persona_split')
exploded['persona_split'] = exploded['persona_split'].str.strip().replace('', 'unknown')
pivot = (
    exploded.groupby(['persona_split', 'tier']).size()
    .unstack(fill_value=0)
    .reindex(columns=['1', '2', '3', 'unknown'], fill_value=0)
)
pivot['total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('total', ascending=False)

col_header = f"  {'Persona/In-Group':<35s}  {'T1':>4}  {'T2':>4}  {'T3':>4}  {'?':>4}  {'tot':>4}"
lines += [col_header, '  ' + '-' * (len(col_header) - 2)]
for persona, row in pivot.iterrows():
    lines.append(
        f"  {str(persona):<35s}  "
        f"{row.get('1', 0):>4d}  {row.get('2', 0):>4d}  "
        f"{row.get('3', 0):>4d}  {row.get('unknown', 0):>4d}  {row['total']:>4d}"
    )

report = '\n'.join(lines)
OUTPUT_SUMMARY.write_text(report, encoding='utf-8')
print(f'Wrote summary → {OUTPUT_SUMMARY}')
print()
print(report)

---

## Methods Table: Glossary Entry Counts by Group and Tier

Three tier-subset views are computed for every reporting group:

| View | Tiers included | Role |
|------|---------------|------|
| **All tiers** | 1 + 2 + 3 | Primary analysis denominator |
| **Tier 1+2** | 1 + 2 | Robustness check denominator (appendix) |
| **Tier 1 only** | 1 | Sensitivity analysis denominator |

Tiers are assigned by description-source domain:
- **Tier 1** — peer-reviewed journals, major outlets, civil-rights watchdogs
- **Tier 2** — advocacy orgs, credible progressive media, subject-matter blogs
- **Tier 3** — crowd-edited wikis (RationalWiki, Wikipedia); high coverage, lower sourcing rigour
- **unknown** — excluded from all three views

Two perspectives are produced:
1. **Pipeline reporting groups** (`taxonomy_level × target` mapped to `report_level × report_target`) — these are the groups that appear in Results. Used to identify sparse-cell caveats.
2. **Raw persona/in-group** — the `persona_in_group` field from the source glossary; used to verify the §3.1 asymmetry claim.

Outputs written:
- `outputs/glossary/glossary_pipeline_groups_by_tier.tsv` — Methods table (pipeline groups)
- `outputs/glossary/glossary_persona_by_tier.tsv` — §3.1 verification table (raw persona)
- `outputs/glossary/glossary_sparse_t12.tsv` — pipeline cells with T1+2 count < 5


In [ ]:
import sys
sys.path.insert(0, str(WORKDIR))

from audit_pipeline.helpers import map_reporting_group, norm_target

# -----------------------------------------------------------------------
# Load the pipeline glossary (taxonomy_level × target) produced by
# data_preprocessing/06_apply_annotations.ipynb.
# -----------------------------------------------------------------------
PIPELINE_GLOSSARY_PATH = WORKDIR / 'outputs' / 'unioned_data' / '06_glossary_label_reference.tsv'
if not PIPELINE_GLOSSARY_PATH.exists():
    print(f'Pipeline glossary not found at {PIPELINE_GLOSSARY_PATH}; skipping pipeline-group table.')
    pg = None
else:
    pg = pd.read_csv(PIPELINE_GLOSSARY_PATH, sep='\t', low_memory=False)
    pg['tier_int'] = pd.to_numeric(pg['tier'], errors='coerce')

    def _report_group(row):
        rg = map_reporting_group(row['taxonomy_level'], row['target'])
        return pd.Series({'report_level': rg.report_level,
                          'report_target': rg.report_target,
                          'report_include': rg.include})

    pg[['report_level', 'report_target', 'report_include']] = pg.apply(_report_group, axis=1)
    pg_inc = pg[pg['report_include']].copy()


# -----------------------------------------------------------------------
# 1. Pipeline reporting-group × tier Methods table
# -----------------------------------------------------------------------
def _count_unique_dw(df, tiers=None):
    if tiers is not None:
        df = df[df['tier_int'].isin(tiers)]
    else:
        df = df[df['tier_int'].notna()]
    return df.groupby(['report_level', 'report_target'])['dogwhistle'].nunique()


if pg is not None:
    t_all = _count_unique_dw(pg_inc).rename('n_all')
    t12   = _count_unique_dw(pg_inc, {1, 2}).rename('n_t12')
    t1    = _count_unique_dw(pg_inc, {1}).rename('n_t1')
    t3    = _count_unique_dw(pg_inc, {3}).rename('n_t3')

    pipeline_table = pd.concat([t_all, t12, t1, t3], axis=1).fillna(0).astype(int).reset_index()
    pipeline_table['t3_pct'] = (
        pipeline_table['n_t3'] / pipeline_table['n_all'].where(pipeline_table['n_all'] > 0) * 100
    ).round(1)
    pipeline_table['t3_majority'] = pipeline_table['t3_pct'] > 50
    pipeline_table['sparse_t12'] = pipeline_table['n_t12'] < 5
    pipeline_table = pipeline_table.sort_values('n_all', ascending=False).reset_index(drop=True)

    out_pipeline = OUTPUT_DIR / 'glossary_pipeline_groups_by_tier.tsv'
    pipeline_table.to_csv(out_pipeline, sep='\t', index=False)
    print(f'Wrote pipeline-group table ({len(pipeline_table)} rows) → {out_pipeline}')

    # Sparse-cell flag table
    sparse = pipeline_table[pipeline_table['sparse_t12']].copy()
    out_sparse = OUTPUT_DIR / 'glossary_sparse_t12.tsv'
    sparse.to_csv(out_sparse, sep='\t', index=False)
    print(f'Wrote sparse-cell table ({len(sparse)} cells with T1+2 < 5) → {out_sparse}')

    print('\n=== PIPELINE GROUPS — ENTRY COUNTS BY TIER SUBSET ===')
    print(pipeline_table[['report_level','report_target','n_all','n_t12','n_t1','n_t3','t3_pct','t3_majority','sparse_t12']].to_string(index=False))

    print('\n=== CELLS WITH T1+2 COUNT < 5 (unreliable in robustness check) ===')
    print(sparse[['report_level','report_target','n_all','n_t12','n_t1','t3_pct']].to_string(index=False))


# -----------------------------------------------------------------------
# 2. Raw persona/in-group × tier table (§3.1 verification)
#    Uses the 340-entry raw glossary already in memory as glossary_df.
# -----------------------------------------------------------------------

# Explode compound persona labels (e.g. "racist / antisemitic")
raw = glossary_df.copy()
raw['tier_int'] = pd.to_numeric(raw['tier'], errors='coerce')
raw['persona_split'] = raw['persona_in_group'].str.split(' / ')
raw = raw.explode('persona_split')
raw['persona_split'] = raw['persona_split'].str.strip().replace('', 'other/unknown')

def _count_terms(df, tiers=None):
    if tiers is not None:
        df = df[df['tier_int'].isin(tiers)]
    else:
        df = df[df['tier_int'].notna()]
    return df.groupby('persona_split')['term'].nunique()

p_all = _count_terms(raw).rename('n_all')
p12   = _count_terms(raw, {1, 2}).rename('n_t12')
p1    = _count_terms(raw, {1}).rename('n_t1')
p3    = _count_terms(raw, {3}).rename('n_t3')

persona_table = pd.concat([p_all, p12, p1, p3], axis=1).fillna(0).astype(int).reset_index()
persona_table.rename(columns={'persona_split': 'persona_in_group'}, inplace=True)
persona_table['t3_pct'] = (
    persona_table['n_t3'] / persona_table['n_all'].where(persona_table['n_all'] > 0) * 100
).round(1)
persona_table = persona_table.sort_values('n_all', ascending=False).reset_index(drop=True)

out_persona = OUTPUT_DIR / 'glossary_persona_by_tier.tsv'
persona_table.to_csv(out_persona, sep='\t', index=False)
print(f'\nWrote persona-group table ({len(persona_table)} rows) → {out_persona}')

print('\n=== RAW PERSONA GROUPS — ENTRY COUNTS BY TIER SUBSET (§3.1 verification) ===')
print(persona_table.to_string(index=False))

# -----------------------------------------------------------------------
# §3.1 asymmetry check: is "transphobic and white supremacist more
# densely represented" still accurate after checking by tier?
# -----------------------------------------------------------------------
print('\n=== §3.1 ASYMMETRY CHECK ===')
major = persona_table[persona_table['n_all'] >= 10].copy()
print('Groups with ≥10 entries at all-tiers, ranked by n_all:')
print(major[['persona_in_group','n_all','n_t12','n_t1','t3_pct']].to_string(index=False))

print('\nAll-tiers ranking (top 5):', ', '.join(
    f"{r['persona_in_group']} ({r['n_all']})" for _, r in persona_table.head(5).iterrows()))
print('T1+2 ranking (top 5):', ', '.join(
    f"{r['persona_in_group']} ({r['n_t12']})"
    for _, r in persona_table.sort_values('n_t12', ascending=False).head(5).iterrows()))
print('T1-only ranking (top 5):', ', '.join(
    f"{r['persona_in_group']} ({r['n_t1']})"
    for _, r in persona_table.sort_values('n_t1', ascending=False).head(5).iterrows()))

t_row = persona_table[persona_table['persona_in_group']=='transphobic'].iloc[0]
ws_row = persona_table[persona_table['persona_in_group']=='white supremacist'].iloc[0]
print(f'\ntransphobic: n_all={t_row.n_all}, n_t12={t_row.n_t12}, n_t1={t_row.n_t1}, T3%={t_row.t3_pct}%')
print(f'white supremacist: n_all={ws_row.n_all}, n_t12={ws_row.n_t12}, n_t1={ws_row.n_t1}, T3%={ws_row.t3_pct}%')


---

## Findings: Caveats, Flags, and §3.1 Prose Update

### Methods table (to go in paper §3.x)

The table `glossary_pipeline_groups_by_tier.tsv` contains `n_all / n_t12 / n_t1` for every pipeline reporting group. Columns `t3_majority` and `sparse_t12` are editorial flags, not for the published table.

**Recommended Methods table header:**

| Reporting group | Level | n (all tiers) | n (T1+2) | n (T1 only) | T3 % |
|---|---|---|---|---|---|
| Trans/NB | lgbtq | 76 | 12 | 2 | 84% |
| Jewish | religion | 72 | 43 | 31 | 40% |
| Black | race | 51 | 36 | 15 | 29% |
| Muslim | religion | 17 | 13 | 11 | 24% |
| Men | gender | 13 | 11 | 7 | 15% |
| Liberal | politics | 13 | 7 | 6 | 46% |
| Middle Eastern | race | 12 | 9 | 7 | 25% |
| Latinx | race | 11 | 7 | 4 | 36% |
| LGB | lgbtq | 9 | 4 | 4 | 56% |
| Women | gender | 7 | 4 | 2 | 43% |
| *remaining 13 groups* | — | ≤4 | ≤2 | ≤2 | — |

---

### Tier 3 concentration caveat

Groups where **>50% of all-tier entries are Tier 3** (i.e., their glossary coverage rests primarily on RationalWiki/Wikipedia documentation):

| Reporting group | n_all | T3 % | n_t12 |
|---|---|---|---|
| Trans/NB | 76 | **84%** | 12 |
| LGB | 9 | 56% | 4 |
| Immigrant (origin) | 4 | 75% | 1 |
| Conservative (politics) | 3 | 100% | 0 |
| Asian (race) | 3 | 67% | 1 |
| White (race) | 2 | 100% | 0 |
| Disability/unspecific | 1 | 100% | 0 |
| Communist/Libertarian/Democrat | 1 each | 100% | 0 |

**Caveat text for those cells' estimates:**
> Presence-rate and annotation-quality estimates for [group] should be interpreted cautiously: the majority of glossary entries for this group are sourced from crowd-edited wikis (Tier 3), which provide broad but lower-rigour coverage. The Tier 1+2 robustness check (Appendix) uses a substantially smaller denominator for this group (n_t12 = [N]), and estimates in that check may be unstable.

---

### Sparse-cell flag: T1+2 count < 5

**15 of 23 pipeline reporting cells have n_t12 < 5.** These cells produce unreliable presence rates in the T1+2 robustness check even if stable in the primary analysis. They should be:
- Flagged in the Appendix robustness table with a "†" or "n<5" marker
- Excluded from any T1+2-based DI ratio comparisons (DI ratio is undefined or unstable when denominator < 5)

Cells with n_t12 = 0: conservative (politics), white (race), disability/unspecific, libertarian, communist, democrat. These groups have **no robustness-check glossary coverage at all** and should be omitted from the Appendix comparison table entirely, not just flagged.

---



In [ ]:
# --- Group metrics summary ---
group_cols = ['persona_in_group', 'type', 'register']
group_metrics = (
    glossary_df.groupby(group_cols, dropna=False, as_index=False)
    .agg(
        term_count=('term', 'count'),
        unique_term_count=('term', 'nunique'),
        total_examples=('example_count', 'sum'),
        avg_examples_per_term=('example_count', 'mean'),
    )
    .sort_values(['total_examples', 'term_count'], ascending=False)
)
group_metrics['avg_examples_per_term'] = group_metrics['avg_examples_per_term'].round(3)

print(f'{len(group_metrics):,} persona/type/register groups')
group_metrics.head(15)


In [ ]:
# --- Example-share pie chart by persona/in-group ---
examples_by_group = (
    glossary_df.groupby('persona_in_group', dropna=False)['example_count']
    .sum()
    .sort_values(ascending=False)
)

examples_by_group.index = [
    idx if isinstance(idx, str) and idx.strip() else 'Unknown'
    for idx in examples_by_group.index
]

plt.figure(figsize=(9, 9))
plt.pie(
    examples_by_group.values,
    labels=examples_by_group.index,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
)
plt.title('Share of Example Contexts by Persona/In-Group')
plt.tight_layout()
plt.show()

examples_by_group.to_frame('example_count')
